<a href="https://colab.research.google.com/github/SBethune103/virtual-running-coach-pipeline/blob/dev/notebooks/05_papers_for_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q datasets polars pyarrow tqdm feedparser

from pathlib import Path
from datasets import load_dataset
import polars as pl
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings("ignore")

BASE = Path("/content/drive/MyDrive/virtual-running-coach")
PAPERS_DIR = BASE / "data/papers"
PAPERS_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Setup complete")

Mounted at /content/drive
✅ Setup complete


In [ ]:
import requests
import feedparser
import polars as pl
from tqdm.auto import tqdm

PAPERS_DIR = Path("/content/drive/MyDrive/virtual-running-coach/data/papers")
PAPERS_DIR.mkdir(parents=True, exist_ok=True)

# Much more specific queries
queries = [
    'ti:"endurance running"',
    'ti:"marathon training"',
    'ti:"running economy"',
    'ti:"lactate threshold" AND (running OR exercise)',
    'ti:"VO2max" AND running',
    'all:"periodization" AND (running OR endurance)',
    'all:"high-intensity interval training" AND running',
    'all:"Norwegian" AND threshold AND training',
    'all:"Lydiard" AND training',
    'ti:"endurance performance"'
]

def fetch_arxiv(query, max_results=20):
    url = f"http://export.arxiv.org/api/query?search_query={query}&start=0&max_results={max_results}&sortBy=relevance"
    response = requests.get(url, timeout=30)
    feed = feedparser.parse(response.content)

    papers = []
    for entry in feed.entries:
        papers.append({
            "id": entry.id,
            "title": entry.title.replace("\n", " ").strip(),
            "abstract": entry.summary.replace("\n", " ").strip(),
            "published": entry.published,
            "authors": ", ".join([a.name for a in entry.authors]) if hasattr(entry, "authors") else ""
        })
    return papers

all_papers = []
for q in tqdm(queries):
    print(f"Searching: {q}")
    try:
        papers = fetch_arxiv(q, max_results=15)
        print(f"  → found {len(papers)}")
        all_papers.extend(papers)
    except Exception as e:
        print(f"  → error: {e}")

print(f"\nTotal papers fetched: {len(all_papers)}")

  0%|          | 0/10 [00:00<?, ?it/s]

Searching: ti:"endurance running"
  → found 1
Searching: ti:"marathon training"
  → found 0
Searching: ti:"running economy"
  → found 0
Searching: ti:"lactate threshold" AND (running OR exercise)
  → found 2
Searching: ti:"VO2max" AND running
  → found 0
Searching: all:"periodization" AND (running OR endurance)
  → found 15
Searching: all:"high-intensity interval training" AND running
  → found 1
Searching: all:"Norwegian" AND threshold AND training
  → found 0
Searching: all:"Lydiard" AND training
  → found 1
Searching: ti:"endurance performance"
  → found 0

Total papers fetched: 20


In [ ]:
# Remove duplicates by title
df = pl.DataFrame(all_papers)
df = df.unique(subset=["title"])

print(f"After removing duplicates: {df.shape[0]} papers")
print(df.head(3))

# Save
df.write_parquet(PAPERS_DIR / "relevant_papers_small.parquet")
print("✅ Saved to Google Drive")

After removing duplicates: 20 papers
shape: (3, 5)
┌───────────────────┬───────────────────┬───────────────────┬───────────────────┬──────────────────┐
│ id                ┆ title             ┆ abstract          ┆ published         ┆ authors          │
│ ---               ┆ ---               ┆ ---               ┆ ---               ┆ ---              │
│ str               ┆ str               ┆ str               ┆ str               ┆ str              │
╞═══════════════════╪═══════════════════╪═══════════════════╪═══════════════════╪══════════════════╡
│ http://arxiv.org/ ┆ Personalized      ┆ Promotions have   ┆ 2022-07-23T07:13: ┆ Jie Yang, Yilin  │
│ abs/2207.1479…    ┆ Promotion         ┆ been trending …   ┆ 57Z               ┆ Li, Deddy Jobs…  │
│                   ┆ Decisio…          ┆                   ┆                   ┆                  │
│ http://arxiv.org/ ┆ Running Consensus ┆ This thesis       ┆ 2013-05-14T07:39: ┆ Paolo Braca      │
│ abs/1305.3046…    ┆ for Decentra…     

In [ ]:
print(df.select(["title", "published"]).head(5))
print("\nExample abstract:")
print(df[0, "abstract"][:400])

shape: (5, 2)
┌─────────────────────────────────┬──────────────────────┐
│ title                           ┆ published            │
│ ---                             ┆ ---                  │
│ str                             ┆ str                  │
╞═════════════════════════════════╪══════════════════════╡
│ Personalized Promotion Decisio… ┆ 2022-07-23T07:13:57Z │
│ Running Consensus for Decentra… ┆ 2013-05-14T07:39:30Z │
│ Minimizing Running Costs in Co… ┆ 2014-02-20T13:42:09Z │
│ Medical Physics and Biomedical… ┆ 2026-06-26T03:14:03Z │
│ Chasing Autonomy: Dynamic Reta… ┆ 2026-03-26T20:48:33Z │
└─────────────────────────────────┴──────────────────────┘

Example abstract:
Promotions have been trending in the e-commerce marketplace to build up customer relationships and guide customers towards the desired actions. Since incentives are effective to engage customers and customers have different preferences for different types of incentives, the demand for personalized promotion decision 